# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

# Examen ICCD753 — Recuperación de Información: SCIFACT

**Dataset asignado:** `BeIR/scifact` (corpus y consultas) + `BeIR/scifact-qrels` (`test`).  
**Objetivo:** recuperar documentos completos con **TF-IDF + similitud coseno** y **BM25**,
evaluarlos con **Precision@10, Recall@10 y NDCG@10** y exponerlos como servicio web REST.

**Servicio:** `GET /health` y `POST /search`. No se desarrolla interfaz gráfica.

**Reproducibilidad:** Python 3.11, dependencias en `requirements-dev.txt`, misma lógica para notebook y API.
Ejecutar **Run All** y guardar para entregar este notebook con las salidas reales.

## 1. Preparación y carga del corpus

Se utilizan identificadores `_id` originales, sin segmentar los documentos. Se descargan
los subconjuntos `corpus`, `queries` y **solo qrels de `test`**. Los qrels no forman parte de la indexación.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Ejecuta el notebook dentro del proyecto SCIFACT_Examen_RI"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.dataset import prepare_data, load_corpus, load_queries, load_test_qrels
from src.preprocess import preprocess
from src.retrieval import SciFactSearchEngine
from src.evaluation import evaluate_models

counts = prepare_data()
corpus = load_corpus()
queries = load_queries()
test_qrels = load_test_qrels()
print("Filas descargadas:", counts)
print("QIDs evaluables en test:", sum(any(s > 0 for s in rels.values()) for rels in test_qrels.values()))
print("Primer documento:", {"_id": corpus[0]["_id"], "title": corpus[0]["title"], "text_preview": corpus[0]["text"][:150]})
assert sys.version_info[:2] == (3, 11), sys.version
print("Python:", sys.version, "| Ejecutable:", sys.executable)

Filas descargadas: {'corpus': 5183, 'queries': 1109, 'qrels_test': 339}
QIDs evaluables en test: 300
Primer documento: {'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text_preview': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabi'}
Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)] | Ejecutable: C:\Users\Cesarbmm\Desktop\U\R.I\ExamenRI\SCIFACT_Examen_RI\.venv\Scripts\python.exe


## 2. Auditoría y decisiones de preprocesamiento

**Preprocesamiento compartido:** NFKC → minúsculas → tokens alfanuméricos →
stopwords inglesas (se conservan las negaciones) → stemming Porter.

La colección se indexa con `title + text`. El resultado siempre corresponde al documento original.
Se comprueba integridad referencial y unicidad antes de recuperar.

In [2]:
doc_ids = {d["_id"] for d in corpus}
query_ids = set(queries)
assert len(doc_ids) == len(corpus), "Document IDs duplicados"
assert set(test_qrels).issubset(query_ids), "Qrels con consultas faltantes"
assert all(set(rels).issubset(doc_ids) for rels in test_qrels.values()), "Qrels con documentos faltantes"
print("Documentos:", len(corpus), "| Consultas:", len(queries), "| QIDs con qrels test:", len(test_qrels))
print("Texto de prueba:", "Aspirin is NOT preventing some infections")
print("Tokens procesados:", preprocess("Aspirin is NOT preventing some infections"))

Documentos: 5183 | Consultas: 1109 | QIDs con qrels test: 300
Texto de prueba: Aspirin is NOT preventing some infections
Tokens procesados: ['aspirin', 'not', 'prevent', 'infect']


## 3. Construcción del índice invertido, TF-IDF y BM25

El **índice invertido explícito** conecta cada término con los identificadores originales.
TF-IDF usa normalización L2, de manera que el producto punto equivale a coseno.
BM25 usa `k1=1.5`, `b=0.75`. No hay embeddings ni fragmentos.

In [3]:
engine = SciFactSearchEngine(corpus)
print("Documentos indexados:", len(engine.document_ids))
print("Vocabulario:", len(engine.vocabulary))
print("Matriz TF-IDF:", engine.tfidf_matrix.shape, "| entradas no cero:", engine.tfidf_matrix.nnz)
example_term = "aspirin" if "aspirin" in engine.vocabulary else sorted(engine.vocabulary)[0]
print(f"Posting list de '{example_term}' (muestra):", engine.inverted_index[example_term][:10])

Documentos indexados: 5183
Vocabulario: 25964
Matriz TF-IDF: (5183, 25964) | entradas no cero: 448419
Posting list de 'aspirin' (muestra): ['409280', '520579', '3866315', '6407356', '7613033', '9614443', '12130200', '15648443', '17897801', '21958900']


## 4. Recuperación de una consulta externa

Se muestran rankings independientes y se validan formato, orden e IDs de documentos.
El buscador admite consultas nuevas, no solamente las incluidas en el dataset.

In [4]:
external_query = "Aspirin reduces the risk of cardiovascular disease"
for model in ("tfidf", "bm25"):
    results = engine.search(external_query, model=model, top_k=5)
    assert len(results) <= 5
    assert len({r["document_id"] for r in results}) == len(results)
    assert all(r["document_id"] in doc_ids for r in results)
    assert [r["score"] for r in results] == sorted([r["score"] for r in results], reverse=True)
    print("\nMODELO:", model.upper())
    display(pd.DataFrame(results)[["rank", "document_id", "score", "title"]])


MODELO: TFIDF


,rank,document_id,score,title
0,1,7613033,0.554338,Aspirin in the chemoprevention of colorectal n...
1,2,15648443,0.452334,Long-term effect of aspirin on cancer risk in ...
2,3,43334921,0.336438,Association of aspirin and NSAID use with risk...
3,4,39903312,0.331249,A randomized trial of aspirin to prevent color...
4,5,5698494,0.312320,The benefits of statins in people without esta...



MODELO: BM25


,rank,document_id,score,title
0,1,7613033,22.434420,Aspirin in the chemoprevention of colorectal n...
1,2,15648443,16.202394,Long-term effect of aspirin on cancer risk in ...
2,3,33118292,15.070938,Optimal therapy for reduction of lipoprotein(a).
3,4,5698494,14.870693,The benefits of statins in people without esta...
4,5,409280,14.840328,National study of physician awareness and adhe...


## 5. Selección exacta de consultas para evaluación

**No se elige un subconjunto favorable.** Se evalúan todas las consultas con qrels positivos
asignadas a `test`. La tabla permite identificar exactamente cuáles y cuántas se usaron.

In [5]:
test_query_ids = sorted(qid for qid, rels in test_qrels.items() if any(value > 0 for value in rels.values()))
selected_queries = pd.DataFrame([
    {"query_id": qid, "query_text": queries[qid], "relevant_documents": sum(s > 0 for s in test_qrels[qid].values())}
    for qid in test_query_ids
])
print("Total de consultas evaluadas:", len(selected_queries))
display(selected_queries.head(10))
assert len(selected_queries) > 0
assert set(test_query_ids) == set(test_qrels), "Hay consultas test excluidas"
assert len(test_query_ids) == 300
print("Todos los IDs test:", ", ".join(test_query_ids))

Total de consultas evaluadas: 300


,query_id,query_text,relevant_documents
0,1,0-dimensional biomaterials show inductive prop...,1
1,100,All hematopoietic stem cells segregate their c...,1
2,1012,Radioiodine treatment of non-toxic multinodula...,1
3,1014,Rapamycin decreases the concentration of triac...,1
4,1019,Rapid phosphotransfer rates govern fidelity in...,1
5,1020,Rapid up-regulation and higher basal expressio...,1
6,1021,Rapid up-regulation and higher basal expressio...,1
7,1024,Recurrent mutations occur frequently within CT...,1
8,1029,Reduced responsiveness to interleukin-2 in reg...,3
9,1041,Replacement of histone H2A with H2A.Z slows ge...,2


Todos los IDs test: 1, 100, 1012, 1014, 1019, 1020, 1021, 1024, 1029, 1041, 1049, 1062, 1086, 1088, 1089, 1099, 1100, 1104, 1107, 1110, 1121, 113, 1130, 1132, 1137, 1140, 1144, 1146, 115, 1150, 1163, 1175, 1179, 118, 1180, 1185, 1187, 1191, 1194, 1196, 1197, 1199, 1200, 1202, 1204, 1207, 1213, 1216, 1221, 1225, 1226, 1232, 124, 1241, 1245, 1259, 1262, 1266, 127, 1270, 1271, 1272, 1273, 1274, 1278, 1279, 128, 1280, 1281, 1282, 129, 1290, 1292, 1298, 13, 130, 1303, 1316, 1319, 132, 1320, 133, 1332, 1335, 1336, 1337, 1339, 1344, 1352, 1359, 1362, 1363, 1368, 137, 1370, 1379, 1382, 1385, 1389, 1395, 141, 142, 143, 146, 148, 163, 171, 179, 180, 183, 185, 198, 208, 212, 213, 216, 217, 218, 219, 230, 232, 233, 236, 237, 238, 239, 248, 249, 261, 268, 269, 274, 275, 279, 294, 295, 298, 3, 300, 303, 312, 314, 324, 327, 338, 343, 350, 354, 36, 362, 380, 384, 385, 386, 388, 399, 410, 411, 415, 42, 421, 431, 436, 437, 439, 440, 443, 452, 475, 478, 48, 49, 491, 5, 50, 501, 502, 507, 508, 51, 513, 51

## 6. Evaluación reproducible: P@10, Recall@10 y NDCG@10

**Métricas macro:** primero se calcula la métrica para cada consulta de `test`
y después su media aritmética. P@10 siempre usa denominador 10, incluso si hay menos
relevantes. NDCG@10 tiene en cuenta el orden y los grados de relevancia.

In [6]:
summary, per_query = evaluate_models(engine, queries, test_qrels)
metrics_df = pd.DataFrame(summary).set_index("model")
metrics_by_query_df = pd.DataFrame(per_query)
print("Resumen de métricas en TEST:")
display(metrics_df.style.format({
    "precision_at_10": "{:.4f}", "recall_at_10": "{:.4f}", "ndcg_at_10": "{:.4f}"
}))
print("Primeras métricas por consulta:")
display(metrics_by_query_df.head(8))
assert set(metrics_df.index) == {"bm25", "tfidf"}
assert metrics_df["queries_evaluated"].nunique() == 1
assert set(metrics_by_query_df.query_id) == set(test_qrels)
assert len(metrics_by_query_df) == 2 * len(test_qrels)
assert not metrics_by_query_df.duplicated(["query_id", "model"]).any()
assert (metrics_df["queries_evaluated"] == len(test_qrels)).all()

Resumen de métricas en TEST:


,queries_evaluated,precision_at_10,recall_at_10,ndcg_at_10
model,,,,
tfidf,300,0.0830,0.7537,0.6051
bm25,300,0.0907,0.8202,0.6844


Primeras métricas por consulta:


,query_id,model,precision_at_10,recall_at_10,ndcg_at_10
0,1,tfidf,0.0,0.0,0.0
1,1,bm25,0.0,0.0,0.0
2,100,tfidf,0.1,1.0,0.5
3,100,bm25,0.1,1.0,1.0
4,1012,tfidf,0.1,1.0,1.0
5,1012,bm25,0.1,1.0,1.0
6,1014,tfidf,0.1,1.0,1.0
7,1014,bm25,0.1,1.0,1.0


## 7. Comparación e interpretación

Se calcula **BM25 − TF-IDF** en cada métrica. El signo del diferencial se interpreta
sin presuponer cuál modelo gana. Debido a que SCIFACT suele tener pocos documentos
relevantes por consulta, P@10 puede ser baja incluso para rankings útiles.

In [7]:
delta = metrics_df.loc["bm25", ["precision_at_10", "recall_at_10", "ndcg_at_10"]] - metrics_df.loc["tfidf", ["precision_at_10", "recall_at_10", "ndcg_at_10"]]
print("Diferencia BM25 - TF-IDF:")
display(delta.to_frame(name="delta"))
for metric, value in delta.items():
    winner = "BM25" if value > 0 else "TF-IDF" if value < 0 else "empate"
    print(f"{metric}: {winner} (Δ={value:+.4f})")
print("Interpretación: BM25 normaliza la frecuencia por longitud documental; "
      "TF-IDF compara vectores L2. Las diferencias observadas son descriptivas, no significancia estadística.")

Diferencia BM25 - TF-IDF:


,delta
precision_at_10,0.007667
recall_at_10,0.066500
ndcg_at_10,0.079296


precision_at_10: BM25 (Δ=+0.0077)
recall_at_10: BM25 (Δ=+0.0665)
ndcg_at_10: BM25 (Δ=+0.0793)
Interpretación: BM25 normaliza la frecuencia por longitud documental; TF-IDF compara vectores L2. Las diferencias observadas son descriptivas, no significancia estadística.


## 8. Verificación de fórmulas, IDs y métricas en todas las consultas test

Se contrasta TF-IDF con `cosine_similarity` y BM25 con su fórmula calculada
desde los postings y frecuencias. Se revisan los 600 rankings de test:
IDs originales, títulos y snippets del corpus, puntuaciones finitas y positivas,
orden descendente, desempates por ID, ausencia de duplicados y ranks consecutivos.
P@10, Recall@10 y NDCG@10 se recalculan por separado para cada consulta y modelo.

In [8]:
import math
import re
from collections import Counter
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

documents_by_id = {doc["_id"]: doc for doc in corpus}
def validate_results(results, top_k):
    assert len(results) <= top_k
    ids = [row["document_id"] for row in results]
    assert len(ids) == len(set(ids))
    assert set(ids) <= doc_ids
    assert [row["rank"] for row in results] == list(range(1, len(results) + 1))
    assert results == sorted(results, key=lambda row: (-row["score"], row["document_id"]))
    for row in results:
        assert set(row) == {"rank", "document_id", "score", "title", "snippet"}
        assert isinstance(row["document_id"], str)
        assert math.isfinite(row["score"]) and row["score"] > 0
        document = documents_by_id[row["document_id"]]
        assert row["title"] == document["title"]
        assert row["snippet"] == re.sub(r"\s+", " ", document["text"]).strip()[:250]

cosine_scores = cosine_similarity(engine.vectorizer.transform([external_query]), engine.tfidf_matrix)[0]
np.testing.assert_allclose(np.asarray(engine.tfidf_matrix.multiply(engine.tfidf_matrix).sum(axis=1)).ravel(), 1)
for row in engine.search(external_query, model="tfidf", top_k=100):
    np.testing.assert_allclose(row["score"], cosine_scores[engine.document_ids.index(row["document_id"])] , rtol=1e-12)

n = len(corpus)
raw_idf = {term: math.log((n - len(ids) + 0.5) / (len(ids) + 0.5))
           for term, ids in engine.inverted_index.items()}
average_idf = math.fsum(raw_idf.values()) / len(raw_idf)
idf = {term: value if value >= 0 else 0.25 * average_idf for term, value in raw_idf.items()}
lengths = np.array([len(tokens) for tokens in engine.tokenized_documents])
frequencies = [Counter(tokens) for tokens in engine.tokenized_documents]
bm25_reference = np.zeros(n)
for term in preprocess(external_query):
    tf = np.array([freq[term] for freq in frequencies])
    bm25_reference += idf.get(term, 0) * tf * 2.5 / (tf + 1.5 * (0.25 + 0.75 * lengths / lengths.mean()))
np.testing.assert_allclose(engine.bm25.get_scores(preprocess(external_query)), bm25_reference, rtol=1e-12)

metric_names = ["precision_at_10", "recall_at_10", "ndcg_at_10"]
stored_metrics = metrics_by_query_df.set_index(["query_id", "model"])
checked_rankings = 0
for qid in test_query_ids:
    relevant = {did: rel for did, rel in test_qrels[qid].items() if rel > 0}
    ideal = math.fsum((2 ** rel - 1) / math.log2(rank + 1)
                      for rank, rel in enumerate(sorted(relevant.values(), reverse=True)[:10], 1))
    for model in ("tfidf", "bm25"):
        ranking = engine.search(queries[qid], model=model, top_k=10)
        validate_results(ranking, 10)
        hits = len({row["document_id"] for row in ranking} & relevant.keys())
        dcg = math.fsum((2 ** relevant.get(row["document_id"], 0) - 1) / math.log2(rank + 1)
                        for rank, row in enumerate(ranking, 1))
        reference = [hits / 10, hits / len(relevant), dcg / ideal]
        actual = stored_metrics.loc[(qid, model), metric_names].to_numpy(dtype=float)
        np.testing.assert_allclose(actual, reference, rtol=1e-12, atol=1e-12)
        checked_rankings += 1
assert checked_rankings == 600
print("TF-IDF coseno y fórmula BM25: OK")
print(f"Rankings y métricas comprobados: {checked_rankings}; consultas test: {len(test_query_ids)}")

TF-IDF coseno y fórmula BM25: OK
Rankings y métricas comprobados: 600; consultas test: 300


## 9. Contrato JSON mediante HTTP real en localhost

Se inicia temporalmente Uvicorn en un puerto libre y se comprueban `/health`
y `/search` sobre el corpus real. Se envían las 300 consultas test a ambos modelos
y se comparan sus respuestas con el motor evaluado. También se comprueban una
consulta externa, los límites de `top_k`, consultas desconocidas y entradas inválidas.
El proceso se cierra al terminar. Esta prueba local no acredita una URL pública.

In [9]:
import socket
import subprocess
import time
import httpx

with socket.socket() as free_socket:
    free_socket.bind(("127.0.0.1", 0))
    port = free_socket.getsockname()[1]
base_url = f"http://127.0.0.1:{port}"
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
api_checked = 0
invalid_checked = 0
with (OUT / "uvicorn_verification.log").open("w", encoding="utf-8") as server_log:
    server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", str(port), "--log-level", "warning"],
        cwd=ROOT, stdout=server_log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NO_WINDOW if sys.platform == "win32" else 0,
    )
    try:
        with httpx.Client(base_url=base_url, timeout=30, trust_env=False) as client:
            deadline = time.monotonic() + 120
            while True:
                assert server.poll() is None, (OUT / "uvicorn_verification.log").read_text(encoding="utf-8")
                try:
                    health = client.get("/health")
                    health.raise_for_status()
                    break
                except httpx.ConnectError:
                    if time.monotonic() >= deadline:
                        raise TimeoutError("Uvicorn no respondió en 120 segundos")
                    time.sleep(0.25)
            assert health.headers["content-type"] == "application/json"
            health_json = health.json()
            assert health_json == {"status": "ok", "dataset": "BeIR/scifact", "documents": len(corpus), "models": ["bm25", "tfidf"]}
            print("GET /health:", health.status_code, health_json)
            for qid in test_query_ids:
                for model in ("tfidf", "bm25"):
                    response = client.post("/search", json={"query": queries[qid], "model": model, "top_k": 10})
                    assert response.status_code == 200
                    assert response.headers["content-type"] == "application/json"
                    body = response.json()
                    assert set(body) == {"model", "results"} and body["model"] == model
                    validate_results(body["results"], 10)
                    assert body["results"] == engine.search(queries[qid], model=model, top_k=10)
                    api_checked += 1
            for model in ("tfidf", "bm25"):
                for top_k in (1, 10, 100):
                    response = client.post("/search", json={"query": external_query, "model": model, "top_k": top_k})
                    assert response.status_code == 200
                    validate_results(response.json()["results"], top_k)
                    assert len(response.json()["results"]) == top_k
                unknown = client.post("/search", json={"query": "qzxynotaword", "model": model})
                assert unknown.status_code == 200 and unknown.json() == {"model": model, "results": []}
            defaults = client.post("/search", json={"query": external_query})
            assert defaults.status_code == 200 and defaults.json()["model"] == "bm25"
            assert len(defaults.json()["results"]) == 10
            invalid_payloads = [{}, {"query": ""}, {"query": "   "}, {"query": 123}, {"query": "a" * 2001},
                                {"query": "test", "model": "other"}]
            invalid_payloads += [{"query": "test", "top_k": value} for value in (0, -1, 101, True, False, "10", 10.0, 1.5, None)]
            for payload in invalid_payloads:
                invalid = client.post("/search", json=payload)
                assert invalid.status_code == 422 and isinstance(invalid.json()["detail"], list)
                invalid_checked += 1
            smoke = subprocess.run([sys.executable, "-X", "utf8", "-m", "scripts.smoke_test", base_url],
                                   cwd=ROOT, capture_output=True, text=True, encoding="utf-8", timeout=60)
            print(smoke.stdout)
            assert smoke.returncode == 0, smoke.stderr
        assert api_checked == 600
        print(f"HTTP real: {api_checked} rankings test, top_k=1/10/100, consulta externa y {invalid_checked} entradas inválidas: OK")
    finally:
        server.terminate()
        server.wait(timeout=15)

GET /health: 200 {'status': 'ok', 'dataset': 'BeIR/scifact', 'documents': 5183, 'models': ['bm25', 'tfidf']}


GET /health: {'status': 'ok', 'dataset': 'BeIR/scifact', 'documents': 5183, 'models': ['bm25', 'tfidf']}
POST /search (bm25) OK | recuperados: 5 | IDs: ['7613033', '15648443', '33118292', '5698494', '409280']
POST /search (tfidf) OK | recuperados: 5 | IDs: ['7613033', '15648443', '43334921', '39903312', '5698494']
Validación de modelo inválido: 422 OK

HTTP real: 600 rankings test, top_k=1/10/100, consulta externa y 15 entradas inválidas: OK


## 10. Dependencias y pruebas automatizadas en el mismo Python 3.11

`pip check` comprueba las dependencias transitivas. Se verifican los rangos
de ambos archivos requirements y se ejecuta pytest con el intérprete del kernel.
Las versiones exactas y los hashes SHA-256 permiten identificar esta ejecución.

In [10]:
import importlib.metadata
import json
import hashlib
from packaging.requirements import Requirement

installed = {}
for requirements_file in ("requirements.txt", "requirements-dev.txt"):
    for line in (ROOT / requirements_file).read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.startswith(("-", "#")):
            continue
        requirement = Requirement(line)
        version = importlib.metadata.version(requirement.name)
        assert version in requirement.specifier, (requirement.name, version, requirement.specifier)
        installed[requirement.name] = version
display(pd.DataFrame(sorted(installed.items()), columns=["dependencia", "version"]))
for args in (["-m", "pip", "check"], ["-m", "pytest", "-q"]):
    check = subprocess.run([sys.executable, "-X", "utf8", *args], cwd=ROOT,
                           capture_output=True, text=True, encoding="utf-8", timeout=120)
    print(check.stdout)
    assert check.returncode == 0, check.stderr
    if "pytest" in args:
        pytest_output = check.stdout
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(OUT / "environment_python311.txt").write_text(freeze.stdout, encoding="utf-8")
data_hashes = {name: hashlib.sha256((ROOT / "data" / name).read_bytes()).hexdigest()
               for name in ("corpus.jsonl", "queries.jsonl", "qrels_test.jsonl")}
print("SHA-256 de los datos:")
display(pd.Series(data_hashes, name="sha256").to_frame())

,dependencia,version
0,datasets,4.8.5
1,fastapi,0.164.0
2,httpx,0.28.1
3,ipykernel,7.4.0
4,jupyterlab,4.6.6
5,nbconvert,7.17.2
6,nltk,3.10.4
7,numpy,2.4.6
8,pandas,2.3.3
9,pytest,9.2.0


No broken requirements found.



.....................                                                    [100%]
============================== warnings summary ===============================
.venv\Lib\site-packages\fastapi\testclient.py:1
  C:\Users\Cesarbmm\Desktop\U\R.I\ExamenRI\SCIFACT_Examen_RI\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
21 passed, 1 warning in 2.57s



SHA-256 de los datos:


,sha256
corpus.jsonl,a393a49ff5c447468f1dff507d351689f3bd0508f172d1...
queries.jsonl,de093d7844d1dac7357016c436cb8cff47a4e1fd6a7e40...
qrels_test.jsonl,dde6645350406e89f06d93a188effe48ab77ded160a6a9...


## 11. Guardado de evidencia

Se exportan las métricas globales y por consulta en `results/`. El **notebook
con outputs visibles**, guardado tras `Run All`, constituye el entregable principal,
junto a la URL pública del servicio.

In [11]:
import json
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
(OUT / "metrics_test.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
metrics_by_query_df.to_csv(OUT / "metrics_per_query_test.csv", index=False)
selected_queries.to_csv(OUT / "test_queries_evaluated.csv", index=False)
print("Guardado:", [p.name for p in OUT.glob("*") if p.is_file()])
print("Recuerda guardar el notebook con las salidas y comprobar la URL pública antes de entregar.")

from src.dataset import SCIFACT_REVISION, QRELS_REVISION
verification = {
    "python": sys.version,
    "executable": sys.executable,
    "dependencies": installed,
    "dataset_counts": counts,
    "source_revisions": {"BeIR/scifact": SCIFACT_REVISION, "BeIR/scifact-qrels": QRELS_REVISION},
    "data_sha256": data_hashes,
    "test_queries_evaluated": len(test_query_ids),
    "rankings_checked": checked_rankings,
    "http_test_rankings_checked": api_checked,
    "http_invalid_payloads_checked": invalid_checked,
    "health": health_json,
    "metrics": summary,
    "pytest": pytest_output,
    "pending": ["URL pública no desplegada ni verificada", "Aviso de deprecación Starlette/httpx en TestClient; no afecta al HTTP real"],
}
(OUT / "verification.json").write_text(json.dumps(verification, ensure_ascii=False, indent=2), encoding="utf-8")
print("Informe reproducible:", OUT / "verification.json")


Guardado: ['environment_python311.txt', 'metrics_per_query_test.csv', 'metrics_test.json', 'test_queries_evaluated.csv', 'uvicorn_verification.log']
Recuerda guardar el notebook con las salidas y comprobar la URL pública antes de entregar.
Informe reproducible: C:\Users\Cesarbmm\Desktop\U\R.I\ExamenRI\SCIFACT_Examen_RI\results\verification.json


## 12. Despliegue y endpoint público (evidencia de entrega)

- **Web Service (Render):** [https://examenri-14iv.onrender.com](https://examenri-14iv.onrender.com)
- **Health:** [https://examenri-14iv.onrender.com/health](https://examenri-14iv.onrender.com/health)
- **Búsqueda:** `POST https://examenri-14iv.onrender.com/search`
- **Repositorio:** [https://github.com/Cesarbmm/ExamenRI](https://github.com/Cesarbmm/ExamenRI)

Las solicitudes de búsqueda utilizan `Content-Type: application/json` con los campos `query`, `model` (`bm25` o `tfidf`) y `top_k`.
